In [1]:
import optuna

optuna.logging.set_verbosity(optuna.logging.CRITICAL)
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

/Users/anzekravanja/Projects/smart-trader-lean/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
study_name = "BTC-OptG-It500-RwAct-v1"
study = optuna.create_study(
    study_name=study_name,
    # directions=["maximize", "maximize"],
    sampler=optuna.samplers.NSGAIISampler(population_size=8*5, seed=42, mutation_prob=0.1),
    direction="maximize",
    # sampler=optuna.samplers.TPESampler(n_startup_trials=10, seed=42),
    pruner=optuna.pruners.PercentilePruner(
        75.0, n_startup_trials=10, n_warmup_steps=20,
    ),
    storage='sqlite:///backtests/{}.db'.format(study_name),
    load_if_exists=True
)

In [3]:
print(
    f"python qtrader_optimize.py --name {study_name} --num-trials 100 --path {'sqlite:///backtests/{}.db'.format(study_name)}" # --use-dask y
)
print (
    f"python qtrader_optimize_runner.py --n-jobs 6 --name {study_name} --num-trials 500 --path {'sqlite:///backtests/{}.db'.format(study_name)}"
)

python qtrader_optimize.py --name BTC-OptG-It500-RwAct-v1 --num-trials 100 --path sqlite:///backtests/BTC-OptG-It500-RwAct-v1.db
python qtrader_optimize_runner.py --n-jobs 6 --name BTC-OptG-It500-RwAct-v1 --num-trials 500 --path sqlite:///backtests/BTC-OptG-It500-RwAct-v1.db


In [6]:
df = study.trials_dataframe() #.drop(columns=['number', 'datetime_start', 'datetime_complete'])
# df.sort_values(by='value', ascending=False).head(10)# .drop(columns=['number', 'datetime_start', 'datetime_complete'])
df.tail(7)
# df

,number,value,datetime_start,datetime_complete,duration,params_exp_alpha,params_exp_w_inc,params_exp_weighting,params_expl_decay,params_expl_min,params_model_lr,params_n_steps_target_update,params_rl_gamma,state
6,6,NaN,2025-01-14 20:03:35.323238,NaT,NaT,0.425,0.000001,0.525,0.910,0.48,0.000005,1300,0.975,RUNNING
7,7,NaN,2025-01-16 05:18:22.135290,NaT,NaT,0.975,0.000003,0.525,0.915,0.03,0.000005,3450,0.825,RUNNING
8,8,NaN,2025-01-16 05:18:22.325010,NaT,NaT,0.075,0.000003,0.625,0.965,0.08,0.000010,4450,0.990,RUNNING
9,9,NaN,2025-01-16 05:18:22.325026,NaT,NaT,0.125,0.000075,0.650,0.935,0.25,0.000100,1750,0.950,RUNNING
10,10,NaN,2025-01-16 05:18:22.387690,NaT,NaT,0.075,0.000075,0.650,0.910,0.21,0.000100,3950,0.925,RUNNING
11,11,NaN,2025-01-16 05:18:22.559110,NaT,NaT,0.125,0.000001,0.125,0.975,0.16,0.000003,1850,0.990,RUNNING
12,12,NaN,2025-01-16 05:18:22.800097,NaT,NaT,0.700,0.000005,0.725,0.930,0.12,0.000003,2600,0.950,RUNNING


In [ ]:
fig = optuna.visualization.plot_optimization_history(study)
fig.layout.height = 400
fig

In [ ]:
fig = optuna.visualization.plot_intermediate_values(study)
fig.layout.height = 400
fig

In [ ]:
fig = optuna.visualization.plot_pareto_front(study, target_names=['Sharpe', 'PnL'], include_dominated_trials=True)
fig.layout.height = 800
fig

In [ ]:
fig = optuna.visualization.plot_pareto_front(study, target_names=['SQN', 'PnL'], include_dominated_trials=False)
fig.layout.height = 800
fig

In [ ]:
optuna.visualization.plot_optimization_history(study, target=lambda t: t.values[0], target_name='SQN', error_bar=True)

In [ ]:
optuna.visualization.plot_optimization_history(study, target=lambda t: t.values[1], target_name='PnL', error_bar=True)

In [ ]:
optuna.visualization.plot_optimization_history(study, target=lambda t: t.values[2], target_name='log(num_trades)', error_bar=True)

In [ ]:
fig = optuna.visualization.plot_param_importances(study, target=lambda t: t.values[0], target_name='SQN')
fig.layout.height = 600
fig

In [ ]:
fig = optuna.visualization.plot_param_importances(study, target=lambda t: t.values[1], target_name='PnL')
fig.layout.height = 600
fig

In [ ]:
fig = optuna.visualization.plot_param_importances(study, target=lambda t: t.values[2], target_name='log(num_trades)')
fig.layout.height = 600
fig

In [ ]:
fig = optuna.visualization.plot_slice(study, params=["exp_weighting", "n_steps_target_update"], target=lambda t: t.values[2])
fig.layout.height = 600
fig.show()

In [ ]:
from datetime import datetime, timedelta, timezone
from qtrader.environments.bitstamp import BitstampMarketEnv
from qtrader.rlflow.persistence import SQLitePersistenceProvider


pprov = SQLitePersistenceProvider(root='.')

In [ ]:
list(pprov.list(prefix=''))

In [ ]:
trades = pprov.load_dict('BitstampMarketEnv-BTCUSD-Trades')

In [ ]:
trades

In [ ]:
for trade in trades:
    for o in trade:
        print (o)
        o['ts'] = o['datetime']
        
        ts = datetime.fromisoformat(o['datetime'])
        candle_delta = timedelta(days=1)
        delta = ts.replace(tzinfo=timezone.utc).timestamp() % candle_delta.total_seconds()
        cdt_next = datetime.utcfromtimestamp(ts.replace(tzinfo=timezone.utc).timestamp() - delta)
        
        o['datetime'] = (cdt_next - timedelta(seconds=1)).isoformat()

In [ ]:
trades

In [ ]:
pprov.persist_dict('BitstampMarketEnv-BTCUSD-Trades', trades)

In [22]:
import json

name = "BTC-OptG-It500-RwAct-v1-Tr0010"


p = '{"run_type": "EVAL", "hyperparams": {"invest_pct": 0.05, "invest_max": 0.25, "expl_min": 0.21, "expl_decay": 0.91, "n_steps_warmup": 10240, "n_step_update": 2, "n_steps_target_update": 3950, "model_n_layers": 4, "model_fl_size": 128, "model_shape": "flat", "exp_memory_size": 36500000, "exp_mini_batch_size": 32, "exp_weighting": 0.65, "exp_w_inc": 7.5e-05, "exp_alpha": 0.07500000000000001, "model_lr": 0.0001, "model_act_func": "relu", "model_l2_reg": 0, "rl_gamma": 0.925, "rl_reward_type": "position-relative-action-log"}}'
p = json.loads(p)['hyperparams']
plist = []
for k, v in p.items():
    plist.append(f"{k}={v}")

' '.join(plist)

'invest_pct=0.05 invest_max=0.25 expl_min=0.21 expl_decay=0.91 n_steps_warmup=10240 n_step_update=2 n_steps_target_update=3950 model_n_layers=4 model_fl_size=128 model_shape=flat exp_memory_size=36500000 exp_mini_batch_size=32 exp_weighting=0.65 exp_w_inc=7.5e-05 exp_alpha=0.07500000000000001 model_lr=0.0001 model_act_func=relu model_l2_reg=0 rl_gamma=0.925 rl_reward_type=position-relative-action-log'

In [23]:
print (
    f"python qtrader_trainer.py --name {name} --iters 1000 --params {' '.join(plist)}"
)

python qtrader_trainer.py --name BTC-OptG-It500-RwAct-v1-Tr0010 --iters 1000 --params invest_pct=0.05 invest_max=0.25 expl_min=0.21 expl_decay=0.91 n_steps_warmup=10240 n_step_update=2 n_steps_target_update=3950 model_n_layers=4 model_fl_size=128 model_shape=flat exp_memory_size=36500000 exp_mini_batch_size=32 exp_weighting=0.65 exp_w_inc=7.5e-05 exp_alpha=0.07500000000000001 model_lr=0.0001 model_act_func=relu model_l2_reg=0 rl_gamma=0.925 rl_reward_type=position-relative-action-log
